# Data Preparation: PlantVillage Metadata

Builds `label_mapping.json` (39-class canonical labels) and `dataset_index.json` (image paths with train/val/test splits).
Run from project root. Required before any experiment notebooks.

In [ ]:
import json
import random
from pathlib import Path
from collections import defaultdict

try:
    from sklearn.model_selection import train_test_split
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False


## Configuration

In [9]:
import os
current_dir = Path(os.getcwd())
BASE_DIR = current_dir.parent if current_dir.name == "data_labeling" else current_dir
PV_ROOT = BASE_DIR / "data" / "Plant_leaf_diseases_dataset_with_augmentation"
OUTPUT_DIR = BASE_DIR / "metadata"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)


## Build label mapping from PlantVillage folder structure

In [10]:
# Scan PlantVillage folders
class_folders = sorted([d for d in PV_ROOT.iterdir() if d.is_dir()])
print(f"Found {len(class_folders)} class folders.")

for d in class_folders[:5]:
    print("-", d.name)


Found 39 class folders.
- Apple___Apple_scab
- Apple___Black_rot
- Apple___Cedar_apple_rust
- Apple___healthy
- Background_without_leaves


In [11]:
def make_canonical_label(folder_name: str) -> str:
    """Convert folder name (e.g. Apple___Apple_scab) to canonical label (apple_apple_scab)."""
    parts = folder_name.split("___")
    if len(parts) == 2:
        crop, disease = parts
        label = f"{crop}_{disease}"
    else:
        label = folder_name
    label = label.replace("(", "").replace(")", "")
    label = label.replace(" ", "_")
    label = label.replace("-", "_")
    return label.lower()

classes = []
id_by_folder = {}

for idx, folder in enumerate(class_folders):
    folder_name = folder.name
    canonical_label = make_canonical_label(folder_name)

    # count images (you can extend to png, jpeg, etc.)
    img_paths = (
        list(folder.glob("*.jpg")) +
        list(folder.glob("*.jpeg")) +
        list(folder.glob("*.png"))
    )

    cls_entry = {
        "id": idx,
        "canonical_label": canonical_label,
        "pv_folders": [folder_name],
        "pv_count": len(img_paths),
        "field_count": 0
    }

    classes.append(cls_entry)
    id_by_folder[folder_name] = idx

print(f"Total classes: {len(classes)}")
classes[:3]


Total classes: 39


[{'id': 0,
  'canonical_label': 'apple_apple_scab',
  'pv_folders': ['Apple___Apple_scab'],
  'pv_count': 1000,
  'field_count': 0},
 {'id': 1,
  'canonical_label': 'apple_black_rot',
  'pv_folders': ['Apple___Black_rot'],
  'pv_count': 1000,
  'field_count': 0},
 {'id': 2,
  'canonical_label': 'apple_cedar_apple_rust',
  'pv_folders': ['Apple___Cedar_apple_rust'],
  'pv_count': 1000,
  'field_count': 0}]

Save label_mapping.json

In [12]:
label_mapping = {
    "classes": classes,
    "meta": {
        "source": "PlantVillage",
        "description": "Unified label mapping for PlantVillage + future field datasets",
        "version": 1
    }
}

label_mapping_path = OUTPUT_DIR / "label_mapping.json"
with open(label_mapping_path, "w") as f:
    json.dump(label_mapping, f, indent=2)

print("Saved label_mapping.json to:", label_mapping_path)


Saved label_mapping.json to: metadata\label_mapping.json


Build dataset_index for PV (paths + split)

In [13]:
# We'll create a JSON list like:
# {
#   "path": "PlantVillage/Apple___Apple_scab/image_001.png",
#   "class_id": 0,
#   "domain": "pv",
#   "split": "train"
# }

# Gather all image paths per class
per_class_images = defaultdict(list)

for folder in class_folders:
    folder_name = folder.name
    class_id = id_by_folder[folder_name]

    img_paths = (
        list(folder.glob("*.jpg")) +
        list(folder.glob("*.jpeg")) +
        list(folder.glob("*.png"))
    )

    for p in img_paths:
        per_class_images[class_id].append(p)

# Build dataset_index with train/val/test splits per class
dataset_index = []

train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

for class_id, img_list in per_class_images.items():
    # Convert to list of paths and sort for reproducibility
    img_list = sorted(img_list)

    if SKLEARN_AVAILABLE and len(img_list) >= 3:
        # Split into train and temp
        img_train, img_temp = train_test_split(
            img_list,
            train_size=train_ratio,
            random_state=RANDOM_SEED,
            shuffle=True
        )
        # Split temp into val and test
        val_size_rel = val_ratio / (val_ratio + test_ratio)
        img_val, img_test = train_test_split(
            img_temp,
            train_size=val_size_rel,
            random_state=RANDOM_SEED,
            shuffle=True
        )
    else:
        # Simple split if sklearn not available
        n = len(img_list)
        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)
        img_train = img_list[:n_train]
        img_val = img_list[n_train:n_train + n_val]
        img_test = img_list[n_train + n_val:]

    def add_records(imgs, split_name):
        for p in imgs:
            dataset_index.append({
                "path": str(p),        # or str(p.relative_to(PV_ROOT.parent)) if you prefer relative paths
                "class_id": class_id,
                "domain": "pv",
                "split": split_name
            })

    add_records(img_train, "train")
    add_records(img_val, "val")
    add_records(img_test, "test")

len(dataset_index)


61486

Save dataset_index.json

In [14]:
dataset_index_path = OUTPUT_DIR / "dataset_index.json"
with open(dataset_index_path, "w") as f:
    json.dump(dataset_index, f, indent=2)

print("Saved dataset_index.json to:", dataset_index_path)


Saved dataset_index.json to: metadata\dataset_index.json


Helper functions for adding field images later